In [ ]:
# Parameters cell. Set defaults here
# day_obs_min = 20250415 # Start of On-sky Commissioning w/ LSSTCam
day_obs_min = 20250620  # Start of SV surveys
day_obs_max = 20250921  # Last night of SV surveys
# day_obs_min = 20251026 # Start of Early Operations period
# day_obs_max = 20260425
# day_obs_max = 20260531 # Now

In [ ]:
SAVE_FIGURES = False
# SURVEY_PROGRAMS = ['BLOCK-365'] # 'BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417', 'BLOCK-419', 'BLOCK-421'] # Commissioning
SURVEY_PROGRAMS = [
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]  # Early Ops
FOCUS_ALIGNMENT_PROGRAMS = ["BLOCK-T539"]

# On-sky Utilization

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from astropy.time import Time
import astropy.units as u

from astroplan import Observer

# %matplotlib widget

In [ ]:
from lsst.utils.plotting import publication_plots

publication_plots.set_rubin_plotstyle()

In [ ]:
import os

os.environ["no_proxy"] += ",.consdb"

from lsst.summit.utils import ConsDbClient

client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")

In [ ]:
instrument = "lsstcam"

In [ ]:
def fetch(day_obs_min, day_obs_max):
    visits_query = f"""
    SELECT 
        *
    FROM
        cdb_{instrument}.visit1 AS visit1 
    WHERE
        airmass > 0.
        AND img_type in ('acq', 'science', 'cwfs', 'focus')
        AND day_obs >= {day_obs_min} AND day_obs <= {day_obs_max}
    """

    visits = client.query(visits_query).to_pandas().sort_values(by="obs_start_mjd")

    return visits

In [ ]:
visits = fetch(day_obs_min, day_obs_max)

In [ ]:
selection = np.isin(
    visits["science_program"], SURVEY_PROGRAMS
)  # & (visits['day_obs'] == 20250415)
print(visits["day_obs"][selection].value_counts().sort_index().to_string())
# visits[selection]['visit_id']
print(np.max(visits["day_obs"][selection].value_counts()))

In [ ]:
observer = Observer.at_site("LSST")

In [ ]:
def dayObsToTime(day_obs):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 00:00:00" % (year, month, day), format="iso")
    return time

In [ ]:
def dayObsToMidnight(day_obs, observer):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 23:59:00" % (year, month, day), scale="utc", format="iso")

    time_midnight = observer.midnight(time, which="nearest")
    return time_midnight

In [ ]:
day_obs_min_time_midnight = dayObsToMidnight(day_obs_min, observer)

offsets = np.arange(
    -1,
    np.floor(
        1.5
        + (dayObsToMidnight(day_obs_max, observer) - day_obs_min_time_midnight)
        .to(u.day)
        .value
    )
    + 1,
)

time_day_obs_twilight_plotting = dayObsToTime(day_obs_min) + (offsets * u.day)
time_evening_astronomical_twilight = observer.twilight_evening_astronomical(
    day_obs_min_time_midnight + offsets * u.day, which="previous"
)
time_morning_astronomical_twilight = observer.twilight_morning_astronomical(
    day_obs_min_time_midnight + offsets * u.day, which="next"
)
time_evening_nautical_twilight = observer.twilight_evening_nautical(
    day_obs_min_time_midnight + offsets * u.day, which="previous"
)
time_morning_nautical_twilight = observer.twilight_morning_nautical(
    day_obs_min_time_midnight + offsets * u.day, which="next"
)

In [ ]:
def convertTime(time_input, time_reference):
    time_output = (time_input - (time_reference + (1.0 * u.day))) * u.day.to(u.hr)
    return time_output.value

In [ ]:
time_exp_midpt_mjd = Time(visits["exp_midpt_mjd"], format="mjd")
time_obs_start_mjd = Time(visits["obs_start_mjd"], format="mjd")
time_obs_end_mjd = Time(visits["obs_end_mjd"], format="mjd")

In [ ]:
plt.figure(figsize=(16, 8), dpi=200)

xticks = np.array(
    [dayObsToTime(day_obs).mjd for day_obs in np.unique(visits["day_obs"])]
)
xtick_labels = np.unique(visits["day_obs"])

day_obs_summary = {
    "day_obs": xtick_labels,
    "day_obs_mjd": xticks,
}
for key in day_obs_summary.keys():
    day_obs_summary[key] = np.array([_ for _ in day_obs_summary[key]])

# Twilight plotting
fill_high = np.tile(16, len(time_day_obs_twilight_plotting))
fill_low = np.tile(-4, len(time_day_obs_twilight_plotting))
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_morning_astronomical_twilight, time_day_obs_twilight_plotting),
    convertTime(time_morning_nautical_twilight, time_day_obs_twilight_plotting),
    alpha=0.2,
    color="black",
    label="-12 deg to -18 deg Twilight",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_morning_nautical_twilight, time_day_obs_twilight_plotting),
    fill_high,
    alpha=0.4,
    color="black",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_evening_nautical_twilight, time_day_obs_twilight_plotting),
    convertTime(time_evening_astronomical_twilight, time_day_obs_twilight_plotting),
    alpha=0.2,
    color="black",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    fill_low,
    convertTime(time_evening_nautical_twilight, time_day_obs_twilight_plotting),
    alpha=0.4,
    color="black",
)

x_engineering = []
y_engineering = []
x_survey = []
y_survey = []
x_focus_alignment = []
y_focus_alignment = []
for day_obs in np.unique(visits["day_obs"]):

    time_day_obs = dayObsToTime(day_obs)

    selection_day_obs = visits["day_obs"] == day_obs

    time_exp_midpt_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    time_obs_start_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    time_obs_end_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    x = np.tile(time_day_obs.mjd, len(visits[selection_day_obs]))

    selection_fbs = np.isin(
        visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
    )
    selection_aos = np.isin(
        visits[selection_day_obs]["science_program"], FOCUS_ALIGNMENT_PROGRAMS
    )

    x_engineering.append(x[~selection_fbs])
    y_engineering.append(time_exp_midpt_mjd[~selection_fbs])
    x_survey.append(x[selection_fbs])
    y_survey.append(time_exp_midpt_mjd[selection_fbs])
    x_focus_alignment.append(x[selection_aos])
    y_focus_alignment.append(time_exp_midpt_mjd[selection_aos])

# All scatter plots at the end
s = 20
plt.scatter(
    np.concatenate(x_engineering),
    np.concatenate(y_engineering),
    marker="_",
    s=s,
    c="tab:orange",
    label="Engineering",
)
# plt.scatter(np.concatenate(x_survey), np.concatenate(y_survey), marker='_', s=s, c='tab:blue', label='Pre-LSST')
plt.scatter(
    np.concatenate(x_survey),
    np.concatenate(y_survey),
    marker="_",
    s=s,
    c="tab:blue",
    label="Survey Programs",
)
plt.scatter(
    np.concatenate(x_focus_alignment),
    np.concatenate(y_focus_alignment),
    marker="_",
    s=s,
    c="tab:green",
    label="Focus and Alignment",
)

plt.legend(loc="upper left", markerscale=2)

plt.xlim(xticks[0] - 1.0, xticks[-1] + 1.0)
plt.ylim(-1.5, 11.0)  # Early Ops
# plt.ylim(-2., 11.) # Commissioning
plt.xticks(xticks, xtick_labels, rotation=90.0, fontsize=8)
plt.ylabel("Time (hrs)")
plt.minorticks_off()
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig("figures/on-sky_utilization_%i-%i.pdf" % (day_obs_min, day_obs_max))
plt.show()

## Number of Pre-LSST Visits Acquired Relative to Ideal Night

In [ ]:
from rubin_nights.dayobs_utils import estimated_baseline_visit_range

In [ ]:
n_vis_expected = []
n_vis_acquired = []
for day_obs in day_obs_summary["day_obs"]:
    n_vis_expected.append(estimated_baseline_visit_range(day_obs)["n_vis_ave"])

    selection_day_obs = visits["day_obs"] == day_obs
    selection_fbs = np.isin(
        visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
    )
    n_vis_acquired.append(np.sum(selection_fbs))

day_obs_summary["n_vis_expected"] = np.array(n_vis_expected)
day_obs_summary["n_vis_acquired"] = np.array(n_vis_acquired)
day_obs_summary = pd.DataFrame(data=day_obs_summary)

In [ ]:
# Not yet checked for weather
day_obs_valid = [
    20260222,
    20260223,
    20260224,
    20260225,
    20260226,
    20260227,
    20260301,
    20260303,
    20260308,
    20260329,
    20260330,
    20260405,
    20260406,
    20260412,
    # 20260516,
    # 20260517,
    # 20260518,
    # 20260511,
    20260525,
    20260526,
    20260528,
    20260529,
    20260530,
]

selection_valid = np.in1d(day_obs_summary["day_obs"], day_obs_valid)


plt.figure(figsize=(16, 8), dpi=200)
selection = day_obs_summary["n_vis_acquired"].values >= 1
plt.scatter(
    xticks[selection & selection_valid],
    day_obs_summary["n_vis_acquired"][selection & selection_valid]
    / day_obs_summary["n_vis_expected"][selection & selection_valid],
    c="tab:blue",
    label="Nights of Full Pre-LSST Observations",
)
plt.scatter(
    xticks[selection & ~selection_valid],
    day_obs_summary["n_vis_acquired"][selection & ~selection_valid]
    / day_obs_summary["n_vis_expected"][selection & ~selection_valid],
    c="0.7",
    label="Other Nights",
)
plt.ylabel("Pre-LSST Visit Acquisition Rate Relative to Ideal Night")
plt.xticks(xticks[selection], xtick_labels[selection], rotation=90.0, fontsize=8)
plt.ylim(0.0, 1.0)
plt.xlim(xticks[0] - 1.0, xticks[-1] + 1.0)
plt.grid()
plt.minorticks_off()
plt.legend(loc="upper left")
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(
        "figures/nvisit_acquired_vs_expected_timeseries_%i-%i.pdf"
        % (day_obs_min, day_obs_max)
    )
plt.show()

In [ ]:
bins = np.linspace(0.0, 1.0, 21)

ratio = (
    day_obs_summary["n_vis_acquired"][selection_valid]
    / day_obs_summary["n_vis_expected"][selection_valid]
)

plt.figure(figsize=(8, 6), dpi=150)
plt.hist(
    ratio, color="tab:blue", alpha=0.8, label="Nights of Full Pre-LSST Observations"
)
# plt.xlabel('Number of Pre-LSST Visits Acquired / Number of Visits for Ideal Night')
plt.xlabel("Pre-LSST Visit Acquisition Rate Relative to Ideal Night")
plt.ylabel("Nights")
plt.axvline(
    np.median(ratio), c="black", ls="--", label="Median = %.2f" % (np.median(ratio))
)
plt.xlim(0.0, 1.0)
plt.legend(loc="upper left")
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(
        "figures/nvisit_acquired_vs_expected_hist_%i-%i.pdf"
        % (day_obs_min, day_obs_max)
    )
plt.show()

## Visit Gap Times 

In [ ]:
visits.columns

In [ ]:
n_gap_time = []
gap_time_low = []
gap_time_50 = []
gap_time_high = []


for day_obs in day_obs_summary["day_obs"]:

    selection_day_obs = visits["day_obs"] == day_obs
    selection_fbs = np.isin(
        visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
    )

    if np.sum(selection_fbs) >= 100:
        gap_times = (
            visits["obs_start_mjd"][selection_day_obs][selection_fbs][1:].values
            - visits["obs_end_mjd"][selection_day_obs][selection_fbs][0:-1].values
        ) * u.day.to(u.second)

        # Apply additional selection for consecutive seq_num values
        selection_consecutive = (
            np.diff(visits["seq_num"][selection_day_obs][selection_fbs]) == 1
        )

        n_gap_time.append(np.sum(selection_consecutive))
        gap_time_low.append(np.percentile(gap_times[selection_consecutive], 25))
        gap_time_50.append(np.median(gap_times[selection_consecutive]))
        gap_time_high.append(np.percentile(gap_times[selection_consecutive], 75))
    else:
        n_gap_time.append(0.0)
        gap_time_low.append(-1.0)
        gap_time_50.append(-1.0)
        gap_time_high.append(-1.0)

day_obs_summary["n_gap_time"] = np.array(n_gap_time)
day_obs_summary["gap_time_low"] = np.array(gap_time_low)
day_obs_summary["gap_time_50"] = np.array(gap_time_50)
day_obs_summary["gap_time_high"] = np.array(gap_time_high)

In [ ]:
plt.figure(figsize=(16, 8), dpi=200)

selection = day_obs_summary["n_gap_time"].values >= 1


plt.errorbar(
    xticks[selection],
    day_obs_summary["gap_time_50"][selection],
    yerr=np.array(
        [
            day_obs_summary["gap_time_50"][selection]
            - day_obs_summary["gap_time_low"][selection],
            day_obs_summary["gap_time_high"][selection]
            - day_obs_summary["gap_time_50"][selection],
        ]
    ),
    fmt="_",
    c="black",
    label="gap time: 25th, 50th, 75th Percentile",
)
plt.axhline(5.0, c="tab:green", ls=":", zorder=0)
plt.axhspan(0.0, 5.0, color="tab:green", alpha=0.1)
plt.ylabel("Visit Gap Time (s)")
plt.xticks(xticks[selection], xtick_labels[selection], rotation=90.0, fontsize=8)
plt.ylim(4.0, 20.0)
plt.xlim(xticks[0] - 1.0, xticks[-1] + 1.0)
plt.grid()
plt.minorticks_off()
plt.legend(loc="upper left")
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig("figures/gap_time_timeseries_%i-%i.pdf" % (day_obs_min, day_obs_max))
plt.show()